[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/12_filtragem_sequencial.ipynb)


# Ajustamento Básico - Filtragem sequencial

**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Diferenciar ajustamento em bloco e estimação sequencial.
2. Apresentar o filtro de Kalman para sistemas lineares.
3. Relacionar predição, atualização e propagação de covariâncias.
4. Introduzir o filtro de Kalman estendido para modelos não lineares.

**Referências:**

- Grewal, M. S., Andrews, A. P., & Bartone, C. G. (2020). *Kalman Filtering: Theory and Practice with MATLAB* (5th ed.). Wiley.
- Gelb, A. (1974). *Applied Optimal Estimation*. MIT Press.


## O Problema

No ajustamento em bloco, todas as observações são processadas de uma vez.

Em aplicações cinemáticas, as observações chegam ao longo do tempo. Nesse caso, é natural atualizar a estimativa a cada nova época.


## 1. Ajustamento em bloco versus sequencial

**Em bloco:** estima todos os parâmetros usando todo o conjunto de observações.

**Sequencial:** atualiza a solução quando uma nova observação chega.

A estimação sequencial é essencial em navegação, rastreamento e monitoramento em tempo real.


## 2. Modelo de estado

O filtro estima um vetor de estado $X_k$ em cada época $k$.

Modelo dinâmico:

$$
X_k = F_k X_{k-1} + w_k
$$

Modelo de observação:

$$
L_k = H_k X_k + e_k
$$


### **2.1 Matrizes de covariância**

- $Q_k$: covariância do ruído do modelo dinâmico.
- $R_k$: covariância do ruído de observação.
- $P_k$: covariância do estado estimado.

A qualidade do filtro depende fortemente da escolha de $Q_k$ e $R_k$.


## 3. Predição

A predição transporta a solução da época anterior para a época atual:

$$
\hat{X}_{k}^{-} = F_k \hat{X}_{k-1}^{+}
$$

$$
P_k^{-} = F_k P_{k-1}^{+}F_k^T + Q_k
$$


## 4. Atualização

Com a observação $L_k$, calcula-se o ganho de Kalman:

$$
K_k = P_k^{-}H_k^T(H_kP_k^{-}H_k^T + R_k)^{-1}
$$

Depois:

$$
\hat{X}_k^{+} = \hat{X}_k^{-} + K_k(L_k - H_k\hat{X}_k^{-})
$$

$$
P_k^{+} = (I-K_kH_k)P_k^{-}
$$


**Exercício 01 (resolvido):** Estime a posição 1D de um móvel com velocidade constante e observações ruidosas de posição.


In [ ]:
import numpy as np

# Estado: X = [posição, velocidade]^T
dt = 1.0
F = np.array([[1, dt], [0, 1]])
H = np.array([[1, 0]])
Q = np.diag([0.01, 0.01])
R = np.array([[0.25]])

z = np.array([0.9, 2.2, 2.8, 4.1, 5.1])
X = np.array([[0.0], [1.0]])
P = np.eye(2)

for k, zk in enumerate(z, start=1):
    X_pred = F @ X
    P_pred = F @ P @ F.T + Q

    innov = np.array([[zk]]) - H @ X_pred
    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T @ np.linalg.inv(S)
    X = X_pred + K @ innov
    P = (np.eye(2) - K @ H) @ P_pred

    print(f"k={k}: pos={X[0,0]:.3f}, vel={X[1,0]:.3f}, sigma_pos={P[0,0]**0.5:.3f}")


## 5. Interpretação geométrica

O filtro combina duas informações:

- a predição do modelo dinâmico;
- a nova observação.

Se $R$ é pequeno, o filtro confia mais na observação. Se $Q$ é pequeno, o filtro confia mais no modelo de movimento.


## 6. Filtro de Kalman Estendido (EKF)

Quando o modelo é não linear:

$$
X_k = f(X_{k-1}) + w_k
$$

$$
L_k = h(X_k) + e_k
$$

usa-se linearização por Jacobianas, de forma análoga ao MMQ não linear.


### **6.1 Conexão com Gauss-Newton**

O EKF pode ser visto como uma linearização local a cada época.

A atualização é parecida com uma etapa de MMQ ponderado, mas incorporando a informação prévia vinda da predição.


## 7. Aplicações em Engenharia Cartográfica

- GNSS cinemático;
- integração GNSS/INS;
- monitoramento estrutural em tempo real;
- rastreamento de veículos;
- navegação de drones;
- séries temporais geodésicas.


**Exercício 02 (resolvido):** Observe o efeito de aumentar a variância da observação $R$.


In [ ]:
for r in [0.04, 0.25, 4.0]:
    R = np.array([[r]])
    X = np.array([[0.0], [1.0]])
    P = np.eye(2)
    for zk in z:
        X_pred = F @ X
        P_pred = F @ P @ F.T + Q
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)
        X = X_pred + K @ (np.array([[zk]]) - H @ X_pred)
        P = (np.eye(2) - K @ H) @ P_pred
    print(f"R={r}: posição final={X[0,0]:.3f}, velocidade final={X[1,0]:.3f}")


## Lista de exercícios complementares

**Exercício 03:** Simule uma trajetória 1D com aceleração e compare um filtro de velocidade constante com um filtro de aceleração constante.

**Exercício 04:** Implemente um EKF para estimar posição 2D a partir de distâncias a estações conhecidas.

**Exercício 05:** Compare ajuste em bloco e filtro sequencial para a mesma série de observações de posição.
